# EDA on raw data: CIC-IDS2017

Looks at the data **before any cleaning**, to decide what the cleaning step must do. Nothing here modifies files.

**About this dataset:** Seven daily files (**Wednesday, the DoS day, is missing**, so Hulk/GoldenEye/slowloris/Slowhttptest are absent). Headers have **leading spaces** (`' Label'`). Sampled at 5% for plots; label counts below are exact.

Run from the `notebooks/` folder (it imports `../src`).

In [ ]:
import os, sys, warnings
# Windows: duplicate OpenMP runtimes corrupt the heap and cause bogus "MemoryError"s. Must be set before numpy/sklearn load.
for _k, _v in {"KMP_DUPLICATE_LIB_OK": "TRUE", "OMP_NUM_THREADS": "1", "MKL_NUM_THREADS": "1", "OPENBLAS_NUM_THREADS": "1"}.items():
    os.environ.setdefault(_k, _v)
sys.path.insert(0, os.path.abspath("../src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.feature_selection import mutual_info_classif
from config import get_config
import eda_utils as eu

DATASET = "cic_ids2017"
SAMPLE_FRAC = 0.05     # fraction of every raw file loaded for the plots (label counts are always exact)

cfg = get_config(DATASET)
LABEL = cfg["label_column"]
MULTI = {"unsw_nb15": "attack_cat"}.get(DATASET, LABEL)      # column holding the attack name
BENIGN_NAMES = {"unsw_nb15": {"normal"}}.get(DATASET, {str(v).lower() for v in cfg["benign_values"]})
NON_FEATURE = ["_source", LABEL] + ([MULTI] if MULTI != LABEL else [])

sns.set_theme(style="whitegrid")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60); pd.set_option("display.max_rows", 120)
print(cfg["display_name"], "-", len(cfg["raw_files"]), "raw files")
for f in cfg["raw_files"]:
    print("  ", os.path.basename(f), f"{os.path.getsize(f)/1e6:,.0f} MB")

## 1. Class distribution (exact, from the full raw files)

In [ ]:
if DATASET == "unsw_nb15":
    _raw = eu.load_sample(DATASET, 1.0)
    per_file = pd.crosstab(_raw["_source"], _raw[MULTI])
    del _raw
else:
    per_file = eu.label_counts(DATASET)

totals = per_file.sum().sort_values(ascending=False)
table = per_file.T.assign(TOTAL=totals).sort_values("TOTAL", ascending=False)
display(table)

names = totals.index.to_series().astype(str).str.strip().str.lower()
is_benign = names.isin(BENIGN_NAMES)
is_junk = names.eq(LABEL.lower()) if (DATASET != "unsw_nb15") else names.eq("__never__")
benign, junk = int(totals[is_benign].sum()), int(totals[is_junk].sum())
attack = int(totals[~is_benign & ~is_junk].sum())
print(f"Benign: {benign:,}   Attack: {attack:,}   ({100*attack/(benign+attack):.2f}% attack)   header rows repeated inside data: {junk:,}")

fig, ax = plt.subplots(1, 2, figsize=(15, max(4, 0.32 * len(totals))))
totals.plot.barh(ax=ax[0], logx=True, color="steelblue"); ax[0].invert_yaxis(); ax[0].set_title("Raw label counts (log scale)")
pf = per_file.T
att_per_file = pf[~(is_benign | is_junk).values].sum()
all_per_file = pf[~is_junk.values].sum()
(att_per_file / all_per_file).sort_values().plot.barh(ax=ax[1], color="firebrick")
ax[1].set_title("Attack share per raw file"); ax[1].set_xlim(0, 1)
plt.tight_layout(); plt.show()

## 2. Load a sample and look at the raw columns

In [ ]:
df = eu.load_sample(DATASET, SAMPLE_FRAC)
print("sample shape:", df.shape, f"| memory {df.memory_usage(deep=True).sum()/1e6:,.0f} MB")
display(df.head(3).T.head(25))
print("dtypes:", df.dtypes.astype(str).value_counts().to_dict())

## 3. Data quality: missing, infinite, text-in-numeric, negatives, constants, duplicates

In [ ]:
# Repeated header rows (label column contains the word 'Label') are junk: remove them from this analysis
junk_mask = df[LABEL].astype(str).str.strip().str.lower().eq(LABEL.lower()) if DATASET != "unsw_nb15" else pd.Series(False, index=df.index)
print("repeated header rows in sample:", int(junk_mask.sum()))
df = df[~junk_mask].reset_index(drop=True)

q = eu.quality_table(df, exclude=NON_FEATURE)
print(f"rows={len(df):,}  features={len(q)}")
print("columns with missing values   :", int((q["missing_%"] > 0).sum()))
print("columns with inf / 'Infinity' :", int((q["inf"] > 0).sum()))
print("columns with text values      :", int((q["non_numeric_text"] > 0).sum()), "| object-dtype columns:", int((q["dtype"] == "object").sum()))
print("columns with negative values  :", int((q["negative"] > 0).sum()))
print("constant columns              :", int((q["nunique"] <= 1).sum()))
print("duplicate rows in sample      :", int(df.drop(columns="_source").duplicated().sum()))

problems = q[(q["missing_%"] > 0) | (q["inf"] > 0) | (q["non_numeric_text"] > 0) | (q["nunique"] <= 1) | (q["dtype"] == "object")]
print("\nColumns with a data-quality issue:")
display(problems)
print("\nMost negative-valued columns (negatives are impossible for counts/durations/sizes, so they signal bad values):")
display(q[q["negative"] > 0].sort_values("negative", ascending=False).head(12))

## 4. Does `config.py` cover what this data needs?

In [ ]:
present = [c for c in cfg["drop_columns"] if c in df.columns]
absent = [c for c in cfg["drop_columns"] if c not in df.columns]
text_cols = list(q.index[q["non_numeric_text"] > 0])   # columns holding real non-numeric text (not just object dtype from stray header rows)
uncovered = [c for c in text_cols if c not in cfg["drop_columns"] and c not in cfg["categorical_columns"]]
print("drop_columns in config     :", cfg["drop_columns"])
print("  found in this data       :", present)
print("  NOT found (harmless)     :", absent)
print("categorical_columns config :", cfg["categorical_columns"])
print("text columns not handled by drop/categorical config:", uncovered, "<- review these")

## 5. Which features carry signal? (mutual information with Attack vs Benign)

In [ ]:
y = eu.to_binary(df[LABEL], cfg["benign_values"])
X = eu.numeric_frame(df, exclude=NON_FEATURE)          # analysis view only; this is not the cleaning step
for c in cfg["categorical_columns"]:
    if c in df.columns:
        X[c] = df[c].astype("category").cat.codes
X = X.drop(columns=[c for c in cfg["drop_columns"] if c in X.columns])

idx = X.sample(min(30000, len(X)), random_state=0).index
mi = pd.Series(mutual_info_classif(X.loc[idx], y.loc[idx], random_state=0), index=X.columns).sort_values(ascending=False)
mi.head(20).iloc[::-1].plot.barh(figsize=(8, 6), color="teal", title="Top 20 features by mutual information (Attack vs Benign)")
plt.tight_layout(); plt.show()
print("features with ~zero mutual information:", int((mi < 1e-3).sum()), "of", len(mi))

## 6. How do the top features separate benign from attack?

In [ ]:
top = list(mi.head(6).index)
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, c in zip(axes.ravel(), top):
    v = np.log1p(X[c].clip(lower=0))
    sns.histplot(x=v, hue=y.map({0: "Benign", 1: "Attack"}), bins=40, stat="density", common_norm=False, element="step", ax=ax)
    ax.set_title(c); ax.set_xlabel("log1p(value)")
plt.tight_layout(); plt.show()

## 7. Redundancy: correlated features and skew

In [ ]:
nz = X.loc[:, X.std() > 0]
corr = nz.corr()
sel = list(mi.head(25).index.intersection(nz.columns))
plt.figure(figsize=(10, 8)); sns.heatmap(corr.loc[sel, sel], cmap="coolwarm", center=0, vmin=-1, vmax=1)
plt.title("Correlation among the 25 most informative features"); plt.tight_layout(); plt.show()

pairs = corr.abs().where(np.triu(np.ones(corr.shape), 1).astype(bool)).stack()
hi = pairs[pairs > 0.9].sort_values(ascending=False)
print(f"feature pairs with |corr| > 0.9: {len(hi)} (the cleaning step drops one column from each)")
display(hi.head(15).to_frame("|corr|"))

skew = nz.skew().abs().sort_values(ascending=False)
print("\nMost skewed features (heavy right tails; trees are fine, LR/MLP/LSTM need scaling):")
display(skew.head(10).to_frame("|skew|"))

## 8. Categorical columns

In [ ]:
if not cfg["categorical_columns"]:
    print("No categorical columns configured for this dataset.")
for c in cfg["categorical_columns"]:
    if c in df.columns:
        rate = y.groupby(df[c]).agg(["mean", "size"]).rename(columns={"mean": "attack_rate", "size": "rows"}).sort_values("rows", ascending=False)
        print(f"{c}: {df[c].nunique()} distinct values")
        display(rate.head(10))

## 9. Summary: what the cleaning step must handle

In [ ]:
constant = list(q.index[q["nunique"] <= 1])
summary = {
    "rows (sample) / features": (len(df), len(q)),
    "attack share": round(float(y.mean()), 4),
    "columns dropped via config": present,
    "constant columns (removed later by the variance filter)": constant,
    "columns with inf/'Infinity'": list(q.index[q["inf"] > 0]),
    "columns with missing values": list(q.index[q["missing_%"] > 0]),
    "text columns not covered by config": uncovered,
    "correlated pairs > 0.9": len(hi),
}
for k, v in summary.items():
    print(f"{k}: {v}")